In [1]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:256"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
!pip install -q huggingface_hub datasets

In [2]:
import gc, json, time, threading, subprocess, psutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from huggingface_hub import hf_hub_download
from datasets import load_dataset
from tqdm.auto import tqdm


try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
    print("[INFO] Google Drive mounted.")
except ImportError:
    IN_COLAB = False
    print("[WARN] Not in Colab.")

_DRIVE_ROOT = "/content/drive/MyDrive/tarecmind"

# DATA_VARIANT_NAME dùng lại cache graph/adj cũ để không phải build lại dữ liệu.
# DATA_VARIANT_NAME và RUN_VARIANT_NAME giữ nguyên như bản warm-up cũ.
# Lưu ý: nếu muốn train lại từ đầu, hãy xóa checkpoint cũ trong SAVE_DIR trước khi chạy.
DATA_VARIANT_NAME = "blair_intralayer_gate_lightgcn"
RUN_VARIANT_NAME  = "blair_intralayer_gate_lightgcn"
VARIANT_NAME      = RUN_VARIANT_NAME

CFG = {
    "REPO_ID":        "chuongdo1104/amazon-2023-gold",
    "SILVER_REPO_ID": "chuongdo1104/amazon-2023-silver",
    "DRIVE_ROOT":     _DRIVE_ROOT,
    "DATA_DIR":       f"{_DRIVE_ROOT}/data_{DATA_VARIANT_NAME}",
    "SAVE_DIR":       f"{_DRIVE_ROOT}/weights_{RUN_VARIANT_NAME}",
    "CACHE_DIR":      "/content/recsys_cache",
    "EMBED_DIM":      128,
    "GCN_LAYERS":     2,
    "EPOCHS":         49,
    "LR_JOINT":       3e-4,
    "L2_REG":        1e-4,
    "WEIGHT_DECAY":   0.0,

    # ===== Popularity-aware negative sampling =====
    # uniform     : sample đều trong warm items như bản cũ.
    # popularity : P(negative item j) ∝ train_freq(j)^NEG_POP_ETA, chỉ trong warm items.
    # NEG_UNIFORM_MIX thêm một phần uniform để không quá lệch hẳn về head item.
    "NEG_SAMPLING":    "popularity",
    "NEG_POP_ETA":     0.50,
    "NEG_UNIFORM_MIX": 0.30,
    "NEG_AVOID_POS":   True,
    "NEG_RESAMPLE_TRIES": 3,

    # ===== Frozen BLaIR + Lightweight Adapter + Intra-layer Gate =====
    "LLM_DIM":         768,
    "ADAPTER_HIDDEN":  256,
    "ADAPTER_DROPOUT": 0.00,
    "TEXT_CHUNK_SIZE": 50000,
    "LAMBDA_ALIGN":    0.02,     # joint phase: trọng số alignment trong loss BPR + align
    "ALIGN_TAU":       0.2,
    "WARMUP_ALIGN_EPOCHS": 3,    # Phase 1: chỉ train L_align trong 1-3 epoch đầu
    "EVAL_DURING_WARMUP": False, # warm-up chưa học ranking nên mặc định không eval/early-stop
    "USE_FINAL_ALPHA": True,
    "USER_BLAIR_PATH": "/content/drive/MyDrive/new_gold_embeddings_blair/gold_user_blair_embeddings.npy",
    "ITEM_BLAIR_PATH": "/content/drive/MyDrive/new_gold_embeddings_blair/gold_item_blair_embeddings.npy",
    "GRAD_CLIP":      1.0,
    "PATIENCE":       5,
    "EVAL_EVERY":     2,
    "KEEPALIVE_MINS": 25,
}

for d in [CFG["DATA_DIR"], CFG["SAVE_DIR"], CFG["CACHE_DIR"]]:
    os.makedirs(d, exist_ok=True)

print(f"[INFO] Data variant: {DATA_VARIANT_NAME}")
print(f"[INFO] Run variant : {RUN_VARIANT_NAME}")
print(f"[INFO] Save dir    : {CFG['SAVE_DIR']}")

Mounted at /content/drive
[INFO] Google Drive mounted.
[INFO] Data variant: blair_intralayer_gate_lightgcn
[INFO] Run variant : blair_intralayer_gate_lightgcn
[INFO] Save dir    : /content/drive/MyDrive/tarecmind/weights_blair_intralayer_gate_lightgcn


In [3]:
def detect_and_adjust_gpu():
    sys_ram_gb = psutil.virtual_memory().total / 1e9
    print(f"[INFO] System RAM: {sys_ram_gb:.1f} GB")
    if not torch.cuda.is_available():
        print("[WARN] No GPU.")
        return "cpu"
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"[INFO] GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB")
    if vram_gb > 70:
        CFG["BATCH_SIZE"]  = 32768
        CFG["ACCUM_STEPS"] = 1
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.backends.cudnn.benchmark = True
    elif vram_gb > 35:
        CFG["BATCH_SIZE"]  = 16384
        CFG["ACCUM_STEPS"] = 2
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.backends.cudnn.benchmark = True
    elif vram_gb > 16:
        CFG["BATCH_SIZE"]  = 4096
        CFG["ACCUM_STEPS"] = 4
    else:
        CFG["BATCH_SIZE"]  = 2048
        CFG["ACCUM_STEPS"] = 4
    print(f"   -> batch={CFG['BATCH_SIZE']}, accum={CFG['ACCUM_STEPS']}")
    return "cuda"

device = detect_and_adjust_gpu()

_keepalive_stop = threading.Event()
def _keepalive_worker():
    interval = CFG["KEEPALIVE_MINS"] * 60
    ka_file  = os.path.join(CFG["CACHE_DIR"], "keepalive.txt")
    while not _keepalive_stop.is_set():
        time.sleep(interval)
        if not _keepalive_stop.is_set():
            with open(ka_file, "w") as f:
                f.write(f"alive {time.strftime('%H:%M:%S')}\n")
threading.Thread(target=_keepalive_worker, daemon=True).start()
print(f"[INFO] Keep-alive started. Device: {device}")

[INFO] System RAM: 189.9 GB
[INFO] GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition | VRAM: 102.0 GB
   -> batch=32768, accum=1
[INFO] Keep-alive started. Device: cuda


In [4]:
POP_GROUP  = {0: "HEAD", 1: "MID", 2: "TAIL", 3: "COLD_START"}
def download_hf(filename, local_dir=None):
    target     = local_dir or CFG["CACHE_DIR"]
    local_path = os.path.join(target, os.path.basename(filename))
    if os.path.exists(local_path):
        return local_path
    print(f"[INFO] Downloading {filename}...")
    return hf_hub_download(
        repo_id=CFG["REPO_ID"], filename=filename,
        repo_type="dataset", local_dir=target,
    )

print("\n--- LOADING GOLD METADATA ---")
with open(download_hf("gold/gold_dataset_stats.json"), "r") as f:
    dataset_stats = json.load(f)

num_users   = dataset_stats["n_users"]
num_items   = dataset_stats["n_items"]
total_nodes = num_users + num_items
print(f"[INFO] {num_users:,} users | {num_items:,} items | {total_nodes:,} total nodes")
print(f"[INFO] Sparsity: {dataset_stats.get('sparsity_pct','N/A')}")

edge_index_raw   = np.load(download_hf("gold/gold_edge_index.npy"))
train_edge_index = torch.from_numpy(edge_index_raw).long()
n_train_edges    = train_edge_index.shape[1]
del edge_index_raw

item_train_freq_np = np.load(download_hf("gold/gold_item_train_freq.npy"))
item_pop_group_np  = np.load(download_hf("gold/gold_item_popularity_group.npy"))
item_train_freq_t  = torch.from_numpy(item_train_freq_np).long()
item_pop_group_t   = torch.from_numpy(item_pop_group_np.astype("int32")).long()

assert n_train_edges > 0,              "Edge list empty!"
assert len(item_train_freq_np) == num_items

# ============================================================
# ALL-ITEM WITH COLD SETUP:
# Warm item = item có ít nhất 1 tương tác trong train.
# Cold item = item có train_freq == 0.
# Evaluation giữ cả warm và cold trong validation/test positives
# và trong candidate ranking.
# Negative sampling khi train vẫn chỉ lấy warm item để không coi
# cold item là negative training signal.
# Tail/Cold được báo cáo riêng để phân tích long-tail và cold-start.
# ============================================================
item_train_freq_cpu = item_train_freq_t.detach().cpu()
warm_item_mask_cpu  = item_train_freq_cpu > 0
cold_item_mask_cpu  = ~warm_item_mask_cpu
warm_item_ids_cpu   = torch.where(warm_item_mask_cpu)[0].long()
num_warm_items      = int(warm_item_mask_cpu.sum().item())
num_cold_items      = int(cold_item_mask_cpu.sum().item())

assert num_warm_items > 0, "No warm items found!"
assert num_warm_items + num_cold_items == num_items

print(f"\n[INFO] {n_train_edges:,} train edges")
for code, name in POP_GROUP.items():
    cnt = int((item_pop_group_t == code).sum())
    warm_cnt = int(((item_pop_group_t == code).cpu() & warm_item_mask_cpu).sum())
    print(f"  {name:10s}: {cnt:,} total | {warm_cnt:,} warm")
cnt_cold = int((item_pop_group_t == 3).sum())
print(f"  {'COLD_START':10s}: {cnt_cold:,} total | 0 warm (freq=0 by definition)")
print(f"[INFO] All-item catalog: {num_items:,} items | warm={num_warm_items:,} | cold kept={num_cold_items:,}")

del item_train_freq_np, item_pop_group_np
gc.collect()
print("[OK] Gold data loaded. Warm/cold masks prepared.")


--- LOADING GOLD METADATA ---
[INFO] Downloading gold/gold_dataset_stats.json...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


gold_dataset_stats.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

[INFO] 1,847,662 users | 1,610,012 items | 3,457,674 total nodes
[INFO] Sparsity: 99.9995%
[INFO] Downloading gold/gold_edge_index.npy...


gold/gold_edge_index.npy:   0%|          | 0.00/223M [00:00<?, ?B/s]

[INFO] Downloading gold/gold_item_train_freq.npy...


gold/gold_item_train_freq.npy:   0%|          | 0.00/12.9M [00:00<?, ?B/s]

[INFO] Downloading gold/gold_item_popularity_group.npy...


gold/gold_item_popularity_group.npy:   0%|          | 0.00/1.61M [00:00<?, ?B/s]


[INFO] 13,964,281 train edges
  HEAD      : 210,925 total | 210,925 warm
  MID       : 116,283 total | 116,283 warm
  TAIL      : 714,913 total | 714,913 warm
  COLD_START: 567,891 total | 0 warm
  COLD_START: 567,891 total | 0 warm (freq=0 by definition)
[INFO] All-item catalog: 1,610,012 items | warm=1,042,121 | cold kept=567,891
[OK] Gold data loaded. Warm/cold masks prepared.


In [5]:
path_train_edges = os.path.join(CFG["DATA_DIR"], "train_edges.pt")
path_val_edges   = os.path.join(CFG["DATA_DIR"], "val_edges.pt")
path_sparse_adj  = os.path.join(CFG["DATA_DIR"], "sparse_adj.pt")

# -- Val edges --
if os.path.exists(path_val_edges):
    print("[INFO] Loading val_edges from Drive cache...")
    val_edges_t = torch.load(path_val_edges, map_location="cpu", weights_only=True)
    VAL_SIZE    = val_edges_t.shape[1]
else:
    print("[INFO] Building val_edges from silver_val_ground_truth.parquet...")
    df_val_gt = load_dataset(
        CFG["SILVER_REPO_ID"],
        data_dir="silver/silver_val_ground_truth.parquet",
        split="train",
    ).to_pandas()
    path_item_map = download_hf("gold/gold_item_id_map.parquet")
    path_user_map = download_hf("gold/gold_user_id_map.parquet")
    df_item_map = pd.read_parquet(path_item_map, columns=["parent_asin", "item_idx"])
    df_user_map = pd.read_parquet(path_user_map, columns=["reviewer_id",  "user_idx"])
    df_val_gt = (
        df_val_gt
        .merge(df_item_map, on="parent_asin", how="inner")
        .merge(df_user_map, on="reviewer_id",  how="inner")
    )
    VAL_SIZE    = len(df_val_gt)
    val_edges_t = torch.tensor(
        np.stack([df_val_gt["user_idx"].values.astype(np.int64),
                  df_val_gt["item_idx"].values.astype(np.int64)], axis=0),
        dtype=torch.long,
    )
    torch.save(val_edges_t, path_val_edges)
    del df_val_gt, df_item_map, df_user_map; gc.collect()
print(f"  val_edges raw: {VAL_SIZE:,}")

# Val positives: giữ nguyên tất cả, kể cả cold items.
# Cold items sẽ xuất hiện trong COLD group khi evaluate.
VAL_SIZE = int(val_edges_t.shape[1])
val_cold_cnt = int((item_train_freq_cpu[val_edges_t[1].cpu()] == 0).sum())
print(f"[INFO] Val: {VAL_SIZE:,} total | cold positives: {val_cold_cnt:,}")

# -- Train edges --
if os.path.exists(path_train_edges):
    train_edge_index_dev = torch.load(path_train_edges, map_location=device, weights_only=True)
else:
    train_edge_index_dev = train_edge_index.to(device)
    torch.save(train_edge_index_dev.cpu(), path_train_edges)
train_edge_index = train_edge_index_dev
n_train_edges    = train_edge_index.shape[1]

# -- Sparse adj (D^-0.5 A D^-0.5) --
if os.path.exists(path_sparse_adj):
    print("[INFO] Loading sparse_adj from Drive cache...")
    sparse_adj = torch.load(path_sparse_adj, map_location=device, weights_only=True)
else:
    print("[INFO] Building sparse_adj...")
    row = torch.cat([train_edge_index[0], train_edge_index[1] + num_users])
    col = torch.cat([train_edge_index[1] + num_users, train_edge_index[0]])
    gei = torch.stack([row, col]).long()
    deg = torch.bincount(gei[0], minlength=total_nodes).float()
    deg_inv_sqrt = deg.pow(-0.5)
    deg_inv_sqrt[torch.isinf(deg_inv_sqrt)] = 0.0
    ew  = (deg_inv_sqrt[gei[0]] * deg_inv_sqrt[gei[1]]).half()
    adj = torch.sparse_coo_tensor(gei, ew, (total_nodes, total_nodes))
    sparse_adj = adj.coalesce().to(device).to_sparse_csr()
    torch.save(sparse_adj.cpu(), path_sparse_adj)
    del adj, ew, deg, deg_inv_sqrt, gei, row, col; gc.collect()

# -- Move tensors and masks to device --
item_train_freq_t = item_train_freq_t.to(device)
item_pop_group_t  = item_pop_group_t.to(device)
warm_item_mask    = warm_item_mask_cpu.to(device)
cold_item_mask    = cold_item_mask_cpu.to(device)
warm_item_ids     = warm_item_ids_cpu.to(device)

print(f"[OK] Graph ready. Train: {n_train_edges:,} | Val: {VAL_SIZE:,} (gồm cả cold)")
print(f"[OK] Candidate set all-item with cold: {num_items:,} items | warm={num_warm_items:,} | cold kept={num_cold_items:,}")
gc.collect(); torch.cuda.empty_cache()

[INFO] Loading val_edges from Drive cache...
  val_edges raw: 1,847,662
[INFO] Val: 1,847,662 total | cold positives: 72,685
[INFO] Loading sparse_adj from Drive cache...
[OK] Graph ready. Train: 13,964,281 | Val: 1,847,662 (gồm cả cold)
[OK] Candidate set all-item with cold: 1,610,012 items | warm=1,042,121 | cold kept=567,891


/usr/local/lib/python3.12/dist-packages/torch/_utils.py:371: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  result = torch.sparse_compressed_tensor(
/usr/local/lib/python3.12/dist-packages/torch/_utils.py:371: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  result = torch.sparse_compressed_tensor(


In [6]:
# ============================================================
# Frozen BLaIR + Lightweight Adapter + Intra-layer Gate Fusion
# Công thức triển khai:
#   gamma_v^(l) = sigmoid(MLP([E_v^(l) || zL_v || deg_v]))
#   E_hat_v^(l) = gamma_v^(l) * E_v^(l) + (1 - gamma_v^(l)) * zL_v
#   E^(l+1)     = A_hat @ E_hat^(l)
# ============================================================

print("\n--- LOADING FROZEN BLaIR EMBEDDINGS ---")
for k in ["USER_BLAIR_PATH", "ITEM_BLAIR_PATH"]:
    if not os.path.exists(CFG[k]):
        raise FileNotFoundError(
            f"Missing {k}: {CFG[k]}\n"
            "Bạn cần merge các part BLaIR thành gold_user_blair.npy / gold_item_blair.npy "
            "hoặc sửa CFG path cho đúng."
        )

user_blair_np = np.load(CFG["USER_BLAIR_PATH"], mmap_mode="r")
item_blair_np = np.load(CFG["ITEM_BLAIR_PATH"], mmap_mode="r")

assert user_blair_np.shape == (num_users, CFG["LLM_DIM"]), \
    f"user_blair shape mismatch: {user_blair_np.shape} vs {(num_users, CFG['LLM_DIM'])}"
assert item_blair_np.shape == (num_items, CFG["LLM_DIM"]), \
    f"item_blair shape mismatch: {item_blair_np.shape} vs {(num_items, CFG['LLM_DIM'])}"

print(f"[OK] user_blair: {user_blair_np.shape} | item_blair: {item_blair_np.shape}")

# Degree feature cho gate: log(1+degree) / log(1+max_degree)
user_deg_np = np.bincount(
    train_edge_index[0].detach().cpu().numpy(),
    minlength=num_users
).astype("float32")
item_deg_np = item_train_freq_cpu.detach().cpu().numpy().astype("float32")

def _log_norm_degree(x):
    x = np.log1p(x).astype("float32")
    mx = float(x.max()) if x.size else 1.0
    return x / max(mx, 1.0)

user_deg_norm_cpu = torch.from_numpy(_log_norm_degree(user_deg_np)).float()
item_deg_norm_cpu = torch.from_numpy(_log_norm_degree(item_deg_np)).float()
node_deg_norm_t = torch.cat([user_deg_norm_cpu, item_deg_norm_cpu], dim=0).to(device)

print(f"[OK] degree feature ready: {tuple(node_deg_norm_t.shape)}")


class TextAdapter(nn.Module):
    """
    Adapter nhẹ: frozen BLaIR 768d -> EMB_DIM.
    BLaIR encoder không train; chỉ adapter này train.
    """
    def __init__(self, in_dim=768, out_dim=128, hidden_dim=512, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
            nn.LayerNorm(out_dim),
        )

    def forward(self, x):
        return self.net(x.float())


class IntraLayerGate(nn.Module):
    """
    Node-wise scalar gate gamma_v^(l).
    Input: [E_v^(l) || zL_v || normalized_degree_v]
    Output: gamma in [0, 1]
    """
    def __init__(self, dim):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(dim * 2 + 1, dim),
            nn.GELU(),
            nn.Linear(dim, 1),
        )

    def forward(self, e_l, z_l, deg_norm):
        if deg_norm.dim() == 1:
            deg_norm = deg_norm.unsqueeze(1)
        x = torch.cat([e_l, z_l, deg_norm.to(e_l.dtype)], dim=1)
        return torch.sigmoid(self.mlp(x))


class BlairIntraLayerFusionLightGCN(nn.Module):
    def __init__(self, num_users, num_items, embed_dim=128, gcn_layers=2,
                 llm_dim=768, adapter_hidden=512, adapter_dropout=0.1,
                 use_final_alpha=True):
        super().__init__()
        self.num_users  = num_users
        self.num_items  = num_items
        self.embed_dim  = embed_dim
        self.gcn_layers = gcn_layers
        self.use_final_alpha = use_final_alpha

        # Graph ID embeddings như LightGCN
        self.user_id_emb = nn.Embedding(num_users, embed_dim)
        self.item_id_emb = nn.Embedding(num_items, embed_dim)
        nn.init.normal_(self.user_id_emb.weight, std=0.01)
        nn.init.normal_(self.item_id_emb.weight, std=0.01)

        # Adapter riêng cho user/item vì text profile khác bản chất
        self.user_adapter = TextAdapter(llm_dim, embed_dim, adapter_hidden, adapter_dropout)
        self.item_adapter = TextAdapter(llm_dim, embed_dim, adapter_hidden, adapter_dropout)

        # Có thể dùng chung gate cho toàn bộ node vì user/item đã phân biệt qua embedding
        self.gate = IntraLayerGate(embed_dim)

        # alpha cuối giống paper: h = alpha*zG + (1-alpha)*zL
        # Khởi tạo alpha khoảng 0.8 để ban đầu vẫn tin graph nhiều hơn.
        init_alpha = 0.60
        init_logit = np.log(init_alpha / (1.0 - init_alpha))
        self.alpha_logit = nn.Parameter(torch.tensor(float(init_logit)))

    def _encode_memmap_in_chunks(self, np_arr, adapter, chunk_size, device, desc="text"):
        """
        Encode toàn bộ frozen BLaIR memmap qua trainable adapter.
        Đây là phần nặng nhất của intra-layer fusion vì zL phải có cho mọi node.
        """
        outs = []
        n = np_arr.shape[0]
        for s in range(0, n, chunk_size):
            e = min(s + chunk_size, n)
            # copy=True để tránh warning read-only memmap khi torch.from_numpy
            x_np = np.array(np_arr[s:e], dtype=np.float32, copy=True)
            x = torch.from_numpy(x_np).to(device, non_blocking=True)
            outs.append(adapter(x))
            del x, x_np
        return torch.cat(outs, dim=0)

    def encode_text_all(self, user_blair_np, item_blair_np, device, chunk_size=50000):
        user_zL = self._encode_memmap_in_chunks(
            user_blair_np, self.user_adapter, chunk_size, device, desc="user_zL"
        )
        item_zL = self._encode_memmap_in_chunks(
            item_blair_np, self.item_adapter, chunk_size, device, desc="item_zL"
        )
        return torch.cat([user_zL, item_zL], dim=0)

    def forward_all(self, adj, user_blair_np, item_blair_np, node_deg_norm,
                    device, chunk_size=50000):
        """
        Exact intra-layer fusion:
            E_hat^(l) = gamma * E^(l) + (1-gamma) * zL
            E^(l+1)   = A_hat @ E_hat^(l)
        Return:
            h_all: final embedding dùng để scoring
            zG_all: layer-averaged language-aware graph embedding
            zL_all: adapted text embedding
            alpha: learned global scalar
        """
        zL_all = self.encode_text_all(user_blair_np, item_blair_np, device, chunk_size)

        E = torch.cat([self.user_id_emb.weight, self.item_id_emb.weight], dim=0)
        layers = [E]

        for _ in range(self.gcn_layers):
            gamma = self.gate(E, zL_all, node_deg_norm)
            E_hat = gamma * E + (1.0 - gamma) * zL_all
            E = torch.sparse.mm(adj, E_hat.to(adj.dtype)).to(torch.float32)
            layers.append(E)

        zG_all = torch.stack(layers, dim=0).mean(dim=0)

        if self.use_final_alpha:
            alpha = torch.sigmoid(self.alpha_logit)
            h_all = alpha * zG_all + (1.0 - alpha) * zL_all
        else:
            alpha = torch.tensor(1.0, device=device)
            h_all = zG_all

        return h_all, zG_all, zL_all, alpha

    def forward(self, adj, batch_users, batch_pos, batch_neg,
                user_blair_np, item_blair_np, node_deg_norm,
                device, chunk_size=50000):
        h_all, zG_all, zL_all, alpha = self.forward_all(
            adj, user_blair_np, item_blair_np, node_deg_norm,
            device=device, chunk_size=chunk_size
        )
        user_h, item_h = h_all.split([self.num_users, self.num_items])
        return user_h[batch_users], item_h[batch_pos], item_h[batch_neg], zG_all, zL_all, alpha

print("[OK] BLaIR intra-layer fusion LightGCN model class defined.")
print("[NOTE] Exact intra-layer fusion nặng hơn post-fusion vì mỗi batch cần zL cho toàn bộ user+item.")



--- LOADING FROZEN BLaIR EMBEDDINGS ---
[OK] user_blair: (1847662, 768) | item_blair: (1610012, 768)
[OK] degree feature ready: (3457674,)
[OK] BLaIR intra-layer fusion LightGCN model class defined.
[NOTE] Exact intra-layer fusion nặng hơn post-fusion vì mỗi batch cần zL cho toàn bộ user+item.


In [8]:
def bpr_loss(user_emb, pos_emb, neg_emb):
    pos_scores = (user_emb * pos_emb).sum(dim=1)   # dot product thô
    neg_scores = (user_emb * neg_emb).sum(dim=1)
    return -F.logsigmoid(pos_scores - neg_scores).mean()


def l2_reg_loss(model, users, pos_items, neg_items):
    """
    Manual L2 regularization.
    Regularize ID embeddings xuất hiện trong BPR batch.
    Adapter/gate đã được regularize gián tiếp qua optimizer nếu muốn thêm weight_decay,
    nhưng hiện tại giữ weight_decay=0.0 để giống setup manual L2.
    """
    user_ego = model.user_id_emb(users)
    pos_ego  = model.item_id_emb(pos_items)
    neg_ego  = model.item_id_emb(neg_items)

    reg = (
        user_ego.norm(2).pow(2)
        + pos_ego.norm(2).pow(2)
        + neg_ego.norm(2).pow(2)
    )
    return 0.5 * reg / users.size(0)


def symmetric_info_nce(z_graph, z_text, tau=0.2):
    """
    Alignment loss giữa graph-view và text-view của cùng node.
    Positive: cùng user/item ở hai view.
    Negative: các node khác trong batch.
    """
    z_graph = F.normalize(z_graph.float(), dim=1)
    z_text  = F.normalize(z_text.float(),  dim=1)
    logits  = torch.matmul(z_graph, z_text.t()) / tau
    labels  = torch.arange(z_graph.size(0), device=z_graph.device)
    return 0.5 * (F.cross_entropy(logits, labels) + F.cross_entropy(logits.t(), labels))





In [9]:
# ============================================================
# Popularity-aware negative sampling
# ============================================================
# Mục tiêu:
# - Khi train vẫn chỉ sample negative từ warm items, không lấy cold items.
# - Nếu NEG_SAMPLING="popularity": item càng phổ biến càng có xác suất được chọn làm negative.
#   Công thức: P(j) ∝ train_freq(j)^NEG_POP_ETA.
# - NEG_UNIFORM_MIX giúp pha thêm một phần uniform để tránh negative toàn head item.


def build_negative_sampler_weights(item_train_freq_t, warm_item_ids, cfg, device):
    """
    Trả về vector xác suất trên warm_item_ids, dùng cho torch.multinomial.

    """
    warm_item_ids = warm_item_ids.to(device)
    freq = item_train_freq_t[warm_item_ids].float().clamp_min(1.0)

    eta = float(cfg.get("NEG_POP_ETA", 0.5))
    weights = freq.pow(eta)

    mix = float(cfg.get("NEG_UNIFORM_MIX", 0.0))
    mix = max(0.0, min(1.0, mix))
    if mix > 0:
        uniform = torch.ones_like(weights)
        # Mix ở mức distribution: (1-mix)*pop_dist + mix*uniform_dist.
        weights = (1.0 - mix) * (weights / weights.sum()) + mix * (uniform / uniform.sum())
    else:
        weights = weights / weights.sum()

    weights = weights.float().contiguous()
    assert torch.isfinite(weights).all(), "Negative sampling weights contain NaN/Inf!"
    assert weights.numel() == warm_item_ids.numel()
    assert weights.sum() > 0
    return weights


neg_sampling_weights = build_negative_sampler_weights(
    item_train_freq_t=item_train_freq_t,
    warm_item_ids=warm_item_ids,
    cfg=CFG,
    device=device,
)


def sample_negatives(batch_users, batch_pos_items, warm_item_ids, device,
                     weights=None, cfg=None):
    """
    Sample negative items cho BPR.

    Parameters
    ----------
    batch_users: Tensor [B]
        Giữ trong API để sau này có thể mở rộng sang user-history rejection.
    batch_pos_items: Tensor [B]
        Positive item của batch. Dùng để tránh neg == pos.
    warm_item_ids: Tensor [num_warm]
        Candidate negative chỉ gồm warm items.
    weights: Tensor [num_warm] hoặc None
        Nếu popularity-aware thì truyền neg_sampling_weights.
    cfg: dict
        Đọc NEG_SAMPLING, NEG_AVOID_POS, NEG_RESAMPLE_TRIES.
    """
    cfg = cfg or CFG
    warm_item_ids = warm_item_ids.to(device)
    batch_pos_items = batch_pos_items.to(device)
    batch_size = int(batch_pos_items.numel())

    mode = str(cfg.get("NEG_SAMPLING", "uniform")).lower()

    if mode == "popularity":
        if weights is None:
            raise ValueError("NEG_SAMPLING='popularity' requires neg_sampling_weights.")
        sample_pos = torch.multinomial(weights.to(device), batch_size, replacement=True)
    elif mode == "uniform":
        sample_pos = torch.randint(0, warm_item_ids.numel(), (batch_size,), device=device)
    else:
        raise ValueError(f"Unknown NEG_SAMPLING={mode}. Use 'uniform' or 'popularity'.")

    neg_items = warm_item_ids[sample_pos]

    # Tránh false negative rõ ràng nhất: neg trùng đúng positive item trong batch.
    # Không reject toàn bộ user history để giữ tốc độ cho dữ liệu lớn.
    if bool(cfg.get("NEG_AVOID_POS", True)):
        tries = int(cfg.get("NEG_RESAMPLE_TRIES", 3))
        for _ in range(max(tries, 0)):
            bad = (neg_items == batch_pos_items)
            if not bad.any():
                break
            n_bad = int(bad.sum().item())
            if mode == "popularity":
                repl_pos = torch.multinomial(weights.to(device), n_bad, replacement=True)
            else:
                repl_pos = torch.randint(0, warm_item_ids.numel(), (n_bad,), device=device)
            neg_items[bad] = warm_item_ids[repl_pos]

    return neg_items


# ---- quick sanity check ----
_test_bsz = 8192
_test_u = train_edge_index[0, :_test_bsz]
_test_p = train_edge_index[1, :_test_bsz]
_neg = sample_negatives(
    _test_u, _test_p,
    warm_item_ids=warm_item_ids,
    device=device,
    weights=neg_sampling_weights,
    cfg=CFG,
)
assert _neg.shape == (_test_bsz,)
assert (_neg >= 0).all() and (_neg < num_items).all()
assert (item_train_freq_t[_neg] > 0).all(), "Negative sampler still returns cold items!"
if bool(CFG.get("NEG_AVOID_POS", True)):
    same_pos_rate = (_neg == _test_p).float().mean().item()
    assert same_pos_rate < 1e-3, f"Too many neg == pos collisions: {same_pos_rate:.6f}"

with torch.no_grad():
    avg_neg_pop = item_train_freq_t[_neg].float().mean().item()
    avg_warm_pop = item_train_freq_t[warm_item_ids].float().mean().item()
    neg_groups = item_pop_group_t[_neg]
    pct_head = (neg_groups == 0).float().mean().item() * 100
    pct_mid  = (neg_groups == 1).float().mean().item() * 100
    pct_tail = (neg_groups == 2).float().mean().item() * 100

print(
    "[OK] Negative sampler ready: "
    f"mode={CFG['NEG_SAMPLING']} | eta={CFG['NEG_POP_ETA']} | mix_uniform={CFG['NEG_UNIFORM_MIX']} | warm-only"
)
print(
    f"     sanity: avg_neg_pop={avg_neg_pop:.1f} vs avg_warm_pop={avg_warm_pop:.1f} | "
    f"HEAD={pct_head:.1f}% MID={pct_mid:.1f}% TAIL={pct_tail:.1f}%"
)
del _test_u, _test_p, _neg


[OK] Negative sampler ready: mode=popularity | eta=0.5 | mix_uniform=0.3 | warm-only
     sanity: avg_neg_pop=90.7 vs avg_warm_pop=13.4 | HEAD=42.6% MID=11.2% TAIL=46.2%


In [10]:
def evaluate_lightgcn(model, adj, item_pop_group_t, eval_groups,
                       num_users, num_items, Ks=(20, 40), device="cuda",
                       user_chunk=64, train_mask_dict=None,
                       warm_item_mask=None):
    """
    Full-ranking INTERACTION-LEVEL validation cho BLaIR intra-layer fusion.

    Candidate: toàn bộ catalog, kể cả cold items.
    Mask: train items bị loại khỏi ranking.
    Embedding dùng scoring là h_all sau intra-layer gate + final alpha.
    """
    model.eval()
    Ks     = sorted(list(Ks))
    maxK   = max(Ks)
    pop_g  = item_pop_group_t.to(device)
    POP_NAMES   = {2: "TAIL", 3: "COLD"}
    group_names = ["OVERALL", "TAIL", "COLD"]

    accum = {
        g: {K: {"hit": 0.0, "ndcg": 0.0, "n": 0} for K in Ks}
        for g in group_names
    }
    recommended_mask = {
        K: torch.zeros(num_items, dtype=torch.bool, device=device) for K in Ks
    }

    with torch.no_grad():
        with torch.amp.autocast("cuda", enabled=(device == "cuda")):
            h_all, zG_all, zL_all, alpha = model.forward_all(
                adj, user_blair_np, item_blair_np, node_deg_norm_t,
                device=device, chunk_size=CFG["TEXT_CHUNK_SIZE"]
            )
        user_final, item_final = h_all.split([num_users, num_items])
        print(f"[EVAL] alpha={float(alpha.detach().cpu()):.4f}")

    mask_value = torch.finfo(torch.float32).min

    for gname, gdata in eval_groups.items():
        if gdata is None:
            continue
        g_u = gdata["u"].to(device)
        g_i = gdata["i"].to(device)
        g_n = len(g_u)

        with torch.no_grad():
            with torch.amp.autocast("cuda", enabled=(device == "cuda")):
                for start in tqdm(range(0, g_n, user_chunk),
                                  desc=f"Eval {gname}", ncols=100, leave=False):
                    end = min(start + user_chunk, g_n)
                    bu  = g_u[start:end]
                    bi  = g_i[start:end]

                    scores = torch.mm(user_final[bu], item_final.t())

                    # Mask train items per user
                    if train_mask_dict is not None:
                        for j, uid in enumerate(bu.cpu().numpy()):
                            mi = train_mask_dict.get(int(uid))
                            if mi is not None:
                                scores[j, mi.to(device)] = mask_value

                    pos_sc = scores[torch.arange(len(bu), device=device), bi]
                    ranks  = (scores > pos_sc.unsqueeze(1)).sum(dim=1) + 1

                    _, topk_idx = torch.topk(scores, maxK, dim=1)
                    for K in Ks:
                        recommended_mask[K][topk_idx[:, :K].reshape(-1)] = True

                    bi_g = pop_g[bi]
                    for K in Ks:
                        hm = (ranks <= K).float()
                        nm = torch.where(
                            ranks <= K,
                            1.0 / torch.log2(ranks.float() + 1.0),
                            torch.zeros_like(ranks.float())
                        )
                        accum["OVERALL"][K]["hit"]  += hm.sum().item()
                        accum["OVERALL"][K]["ndcg"] += nm.sum().item()
                        accum["OVERALL"][K]["n"]    += len(bu)

                        for code, gn in POP_NAMES.items():
                            m = (bi_g == code)
                            if m.any():
                                accum[gn][K]["hit"]  += hm[m].sum().item()
                                accum[gn][K]["ndcg"] += nm[m].sum().item()
                                accum[gn][K]["n"]    += m.sum().item()

    results = {}
    for grp in group_names:
        results[grp] = {}
        for K in Ks:
            n = accum[grp][K]["n"]
            results[grp][f"Recall@{K}"] = accum[grp][K]["hit"]  / max(n, 1)
            results[grp][f"NDCG@{K}"]   = accum[grp][K]["ndcg"] / max(n, 1)
            results[grp][f"n@{K}"]      = n
        k0 = Ks[0]
        results[grp]["Recall@K"] = results[grp][f"Recall@{k0}"]
        results[grp]["NDCG@K"]   = results[grp][f"NDCG@{k0}"]
        results[grp]["n"]        = results[grp][f"n@{k0}"]

    tail_mask_dev = (pop_g == 2)
    n_tail = int(tail_mask_dev.sum().item())
    cov, tcov, apop = {}, {}, {}
    for K in Ks:
        rm  = recommended_mask[K]
        nr  = int(rm.sum().item())
        cov[K]  = nr / num_items if num_items > 0 else 0
        tcov[K] = (rm & tail_mask_dev).sum().item() / n_tail if n_tail > 0 else 0
        apop[K] = item_train_freq_t[rm].float().mean().item() if nr > 0 else 0

    results["Coverage@K"]       = cov[Ks[0]]
    results["TailCoverage@K"]   = tcov[Ks[0]]
    results["AvgPopularity@K"]  = apop[Ks[0]]
    results["CoverageByK"]      = cov
    results["TailCoverageByK"]  = tcov
    results["AvgPopularityByK"] = apop
    results["EvalLevel"]        = "interaction-level"
    return results


def print_eval_results(epoch, results, Ks=(20, 40)):
    print(f"\n[VAL INTERACTION-LEVEL] Epoch {epoch:02d}")
    for grp in ["COLD", "TAIL", "OVERALL"]:
        if grp not in results:
            continue
        v = results[grp]
        n = v.get(f"n@{Ks[0]}", v.get("n", 0))
        parts = [f"  {grp:<10} | n={n:>10,}"]
        for K in Ks:
            parts.append(f"R@{K}={v.get(f'Recall@{K}', 0):.4f} | N@{K}={v.get(f'NDCG@{K}', 0):.4f}")
        print(" | ".join(parts))
    cov  = results.get("CoverageByK",  {})
    tcov = results.get("TailCoverageByK", {})
    apop = results.get("AvgPopularityByK", {})
    for K in Ks:
        print(f"  Coverage@{K}={cov.get(K,0):.4f} | TailCov@{K}={tcov.get(K,0):.4f} | AvgPop@{K}={apop.get(K,0):.1f}")
    tail = results.get("TAIL", {})
    cold = results.get("COLD", {})
    print(f"  [TAIL] R@20={tail.get('Recall@20',0):.4f} N@20={tail.get('NDCG@20',0):.4f} TailCov@20={tcov.get(20,0):.4f}")
    print(f"  [COLD] R@20={cold.get('Recall@20',0):.4f} N@20={cold.get('NDCG@20',0):.4f}")

print("[OK] Interaction-level BLaIR intra-layer fusion evaluation ready.")


[OK] Interaction-level BLaIR intra-layer fusion evaluation ready.


In [11]:
class ColabCheckpointManager:
    def __init__(self, weight_dir, data_dir, K=20):
        self.weight_dir = weight_dir
        self.data_dir   = data_dir
        self.K          = K
        os.makedirs(weight_dir, exist_ok=True)
        os.makedirs(data_dir, exist_ok=True)
        self.best_path = os.path.join(weight_dir, "best.pth")
        self.last_path = os.path.join(weight_dir, "last.pth")
        self.hist_path = os.path.join(data_dir,   "training_history.json")

    def save(self, model, optimizer, epoch, metrics, history_list, cfg, is_best=False):
        def _safe(v):
            return v if isinstance(v, (int,float,str,bool,type(None))) else str(v)
        safe_cfg = {k: _safe(v) for k,v in cfg.items() if not k.startswith("_")}
        payload = {
            "epoch":                epoch,
            "model_state_dict":     model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "metrics":              metrics,
            "config":               safe_cfg,
        }
        torch.save(payload, self.last_path)
        if is_best:
            torch.save(payload, self.best_path)
            print(f"[CKPT] NEW BEST saved (Score={metrics.get('Score',0):.4f})")

        current = []
        if os.path.exists(self.hist_path):
            try:
                with open(self.hist_path, "r") as f: current = json.load(f)
            except Exception: pass
        existing_epochs = {h["epoch"] for h in current}
        for entry in history_list:
            if entry.get("epoch") not in existing_epochs:
                current.append(entry)
        current.sort(key=lambda x: x.get("epoch", 0))
        with open(self.hist_path, "w") as f:
            json.dump(current, f, indent=2, ensure_ascii=False)

    def try_resume(self, model, optimizer, device):
        if not os.path.exists(self.last_path):
            print("[CKPT] No checkpoint found. Starting from epoch 1.")
            return 0, 0.0, []
        ckpt = torch.load(self.last_path, map_location=device, weights_only=True)
        model.load_state_dict(ckpt["model_state_dict"])
        try: optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        except Exception as e: print(f"[WARN] optimizer state: {e}")
        start_epoch = ckpt.get("epoch", 0)
        history = []
        if os.path.exists(self.hist_path):
            try:
                with open(self.hist_path, "r") as f: history = json.load(f)
            except Exception: pass
        scores = [h.get("Score", 0.4*h.get("Recall@K", 0))
                  for h in history if "Score" in h or "Recall@K" in h]
        best_score = max(scores) if scores else 0.0
        print(f"[CKPT] Resume from epoch {start_epoch + 1}, best_score={best_score:.4f}")
        return start_epoch, best_score, history

ckpt_mgr = ColabCheckpointManager(weight_dir=CFG["SAVE_DIR"], data_dir=CFG["DATA_DIR"])
print("[OK] CheckpointManager ready.")

[OK] CheckpointManager ready.


In [12]:
def build_full_eval_groups(val_edges_t):
    """
    Full validation set — keeps original distribution.
    Used for early stopping and best checkpoint selection.
    """
    val_u = val_edges_t[0].cpu()
    val_i = val_edges_t[1].cpu()
    print(f"  [FULL VAL] {len(val_u):,} samples")
    return {"FULL": {"u": val_u, "i": val_i}}

print("[OK] Eval group builder ready.")

[OK] Eval group builder ready.


In [13]:
for var in ["model", "optimizer", "scaler"]:
    if var in globals(): del globals()[var]
gc.collect(); torch.cuda.empty_cache()

model = BlairIntraLayerFusionLightGCN(
    num_users=num_users,
    num_items=num_items,
    embed_dim=CFG["EMBED_DIM"],
    gcn_layers=CFG["GCN_LAYERS"],
    llm_dim=CFG["LLM_DIM"],
    adapter_hidden=CFG["ADAPTER_HIDDEN"],
    adapter_dropout=CFG["ADAPTER_DROPOUT"],
    use_final_alpha=CFG["USE_FINAL_ALPHA"],
).to(device)

scaler = torch.amp.GradScaler("cuda", enabled=(device=="cuda"))
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=CFG["LR_JOINT"],
    weight_decay=0.0,
)
resume_epoch, best_score, history = ckpt_mgr.try_resume(model, optimizer, device)

print(f"[OK] BLaIR Intra-layer Fusion LightGCN params: {sum(p.numel() for p in model.parameters()):,}")
print(f"[OK] lambda_align={CFG['LAMBDA_ALIGN']} | tau={CFG['ALIGN_TAU']} | warmup_epochs={CFG['WARMUP_ALIGN_EPOCHS']} | text_chunk={CFG['TEXT_CHUNK_SIZE']}")
gc.collect(); torch.cuda.empty_cache()


[CKPT] Resume from epoch 15, best_score=0.0119
[OK] BLaIR Intra-layer Fusion LightGCN params: 443,078,530
[OK] lambda_align=0.02 | tau=0.2 | warmup_epochs=3 | text_chunk=50000


In [18]:
print("[INFO] Building full validation groups for checkpoint selection...")
val_full_groups = build_full_eval_groups(val_edges_t)

print("[INFO] Building train mask dict...")
df_edges = pd.DataFrame({
    "u": train_edge_index[0].cpu().numpy(),
    "i": train_edge_index[1].cpu().numpy()
})
train_mask_dict = df_edges.groupby("u")["i"].apply(
    lambda x: torch.tensor(x.values, dtype=torch.long, device="cpu")
).to_dict()
del df_edges
print(f"[OK] Train mask for {len(train_mask_dict):,} users.")
gc.collect(); torch.cuda.empty_cache()

[INFO] Building full validation groups for checkpoint selection...
  [FULL VAL] 1,847,662 samples
[INFO] Building train mask dict...
[OK] Train mask for 1,847,662 users.


In [14]:
CFG["PATIENCE"] = 5

# Do NOT override resume_epoch here — it is set by ckpt_mgr.try_resume().
if resume_epoch == 0:
    best_score = 0.0
    history    = []
patience_cnt = 0

if resume_epoch > 0:
    for g in optimizer.param_groups:
        g.setdefault("initial_lr", g["lr"])

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG["EPOCHS"], eta_min=1e-5, last_epoch=resume_epoch-1
)

warmup_epochs = int(CFG.get("WARMUP_ALIGN_EPOCHS", 0))
eval_during_warmup = bool(CFG.get("EVAL_DURING_WARMUP", False))

steps_per_epoch = (n_train_edges + CFG["BATCH_SIZE"] - 1) // CFG["BATCH_SIZE"]
print(f"\n[INIT] Training BLaIR Intra-layer Gate LightGCN (Epoch {resume_epoch+1}/{CFG['EPOCHS']})")
print("[INFO] Each epoch uses torch.randperm — every train edge visited once per epoch.")
print("[INFO] Two-phase training:")
print(f"       Phase 1 warm-up: epoch 1 → {warmup_epochs}, loss = L_align + L2")
print(f"       Phase 2 joint:   epoch {warmup_epochs+1} → {CFG['EPOCHS']}, loss = BPR + lambda_align*L_align + L2")
print("[WARN] Exact intra-layer fusion is heavy: adapter+gate for all nodes is recomputed each batch.")
print(f"[INFO] Negative sampling: {CFG['NEG_SAMPLING']} | eta={CFG.get('NEG_POP_ETA', 0)} | uniform_mix={CFG.get('NEG_UNIFORM_MIX', 0)} | warm-only=True")

for epoch in range(resume_epoch, CFG["EPOCHS"]):
    t0 = time.time()
    model.train()

    # epoch is 0-based; epoch 0..warmup_epochs-1 are alignment warm-up.
    is_warmup = epoch < warmup_epochs
    phase_name = "ALIGN-WARMUP" if is_warmup else "JOINT-BPR"
    phase_desc = "L_align + L2 only" if is_warmup else "BPR + lambda_align*L_align + L2"

    total_loss = 0.0
    total_bpr = 0.0
    total_align = 0.0
    total_reg = 0.0
    n_processed = 0

    perm = torch.randperm(n_train_edges, device=device)

    with tqdm(total=n_train_edges,
              desc=f"Ep {epoch+1:02d} [{phase_name}]", ncols=100) as pbar:
        optimizer.zero_grad(set_to_none=True)
        pending_accum_steps = 0

        for step, start in enumerate(range(0, n_train_edges, CFG["BATCH_SIZE"])):
            end = min(start + CFG["BATCH_SIZE"], n_train_edges)
            idx = perm[start:end]

            u = train_edge_index[0, idx]
            p = train_edge_index[1, idx]

            # Negative items are still sampled in warm-up only because the current
            # model.forward(...) API returns h_n and L2 uses neg item ID embeddings.
            # BPR is logged in warm-up but is NOT part of the warm-up objective.
            # Popularity-aware mode: P(neg item j) ∝ train_freq(j)^eta, warm items only.
            n = sample_negatives(
                u, p,
                warm_item_ids=warm_item_ids,
                device=device,
                weights=neg_sampling_weights,
                cfg=CFG,
            )

            with torch.amp.autocast("cuda", enabled=(device=="cuda")):
                h_u, h_p, h_n, zG_all, zL_all, alpha = model(
                    sparse_adj, u, p, n,
                    user_blair_np, item_blair_np, node_deg_norm_t,
                    device=device, chunk_size=CFG["TEXT_CHUNK_SIZE"]
                )

                user_zG, item_zG = zG_all.split([num_users, num_items])
                user_zL, item_zL = zL_all.split([num_users, num_items])

                if is_warmup:
                    # BPR is not optimized in Phase 1. no_grad avoids building
                    # extra BPR graph while still letting us log its value.
                    with torch.no_grad():
                        bpr = bpr_loss(h_u, h_p, h_n)
                else:
                    bpr = bpr_loss(h_u, h_p, h_n)

                reg = l2_reg_loss(model, u, p, n)

                # Warm-up always needs alignment even if LAMBDA_ALIGN is set to 0
                # for later ablation. In joint phase, skip it only when lambda=0.
                need_align = is_warmup or (CFG["LAMBDA_ALIGN"] > 0)
                if need_align:
                    # Use unique nodes to avoid treating duplicated ids inside the
                    # mini-batch as false negatives in InfoNCE.
                    u_align = torch.unique(u)
                    p_align = torch.unique(p)
                    align_u = symmetric_info_nce(user_zG[u_align], user_zL[u_align], CFG["ALIGN_TAU"])
                    align_i = symmetric_info_nce(item_zG[p_align], item_zL[p_align], CFG["ALIGN_TAU"])
                    align = align_u + align_i
                else:
                    align = torch.zeros((), device=device)

                if is_warmup:
                    # Phase 1 in paper: only optimize alignment first.
                    # L2 is kept small for stability, but BPR is excluded.
                    raw_loss = align + CFG["L2_REG"] * reg
                else:
                    # Phase 2: joint CF ranking + graph/text alignment.
                    raw_loss = bpr + CFG["LAMBDA_ALIGN"] * align + CFG["L2_REG"] * reg

                loss = raw_loss / CFG["ACCUM_STEPS"]

            scaler.scale(loss).backward()
            pending_accum_steps += 1

            is_accum_boundary = (pending_accum_steps >= CFG["ACCUM_STEPS"])
            is_last_step      = (step == steps_per_epoch - 1)

            if is_accum_boundary or is_last_step:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["GRAD_CLIP"])
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                pending_accum_steps = 0

            bs = len(u)
            total_loss  += raw_loss.item() * bs
            total_bpr   += bpr.item() * bs
            total_align += align.item() * bs
            total_reg   += reg.item() * bs
            n_processed += bs
            pbar.update(bs)

            del h_u, h_p, h_n, zG_all, zL_all, bpr, reg, align, raw_loss, loss

    scheduler.step()
    avg_loss  = total_loss / max(n_processed, 1)
    avg_bpr   = total_bpr / max(n_processed, 1)
    avg_align = total_align / max(n_processed, 1)
    avg_reg   = total_reg / max(n_processed, 1)
    alpha_val = float(torch.sigmoid(model.alpha_logit).detach().cpu())

    print(
        f"[TRAIN] Ep {epoch+1:02d} | Phase={phase_name} | Obj={phase_desc} "
        f"| Loss={avg_loss:.4f} | BPR(log)={avg_bpr:.4f} "
        f"| Align={avg_align:.4f} | Reg={avg_reg:.4f} | alpha={alpha_val:.4f} "
        f"| SeenEdges={n_processed:,}/{n_train_edges:,} | time={(time.time()-t0)/60:.1f}m"
    )

    metrics = {
        "Phase": phase_name,
        "Warmup": is_warmup,
        "TrainLoss": avg_loss,
        "TrainBPR": avg_bpr,
        "TrainAlign": avg_align,
        "TrainReg": avg_reg,
        "Alpha": alpha_val,
    }
    is_best = False

    # Warm-up is alignment-only, so ranking validation/early stopping is skipped
    # unless EVAL_DURING_WARMUP=True.
    is_first_joint_epoch = (warmup_epochs > 0 and epoch == warmup_epochs)
    should_eval = ((epoch + 1) % CFG["EVAL_EVERY"] == 0) or is_first_joint_epoch

    if is_warmup and not eval_during_warmup:
        print("[VAL] Skip validation during alignment warm-up because BPR ranking is not trained yet.")
    elif should_eval:
        print("\n[VAL-FULL] Interaction-level early stopping and best checkpoint selection.")
        res_full = evaluate_lightgcn(
            model, sparse_adj, item_pop_group_t, val_full_groups,
            num_users, num_items, Ks=(20, 40), device=device,
            train_mask_dict=train_mask_dict,
            warm_item_mask=None
        )
        print_eval_results(epoch+1, res_full, Ks=(20, 40))

        overall_ndcg20 = res_full["OVERALL"]["NDCG@20"]
        score          = overall_ndcg20

        metrics.update({
            "EvalLevel":             "interaction-level",
            "Recall@20":             res_full["OVERALL"]["Recall@20"],
            "NDCG@20":               res_full["OVERALL"]["NDCG@20"],
            "Recall@40":             res_full["OVERALL"]["Recall@40"],
            "NDCG@40":               res_full["OVERALL"]["NDCG@40"],
            "TailRecall@20":         res_full.get("TAIL", {}).get("Recall@20", 0),
            "TailNDCG@20":           res_full.get("TAIL", {}).get("NDCG@20", 0),
            "TailRecall@40":         res_full.get("TAIL", {}).get("Recall@40", 0),
            "TailNDCG@40":           res_full.get("TAIL", {}).get("NDCG@40", 0),
            "Score":                 score,
            "Coverage@20":           res_full.get("CoverageByK", {}).get(20, 0),
            "TailCoverage@20":       res_full.get("TailCoverageByK", {}).get(20, 0),
            "AvgPopularity@20":      res_full.get("AvgPopularityByK", {}).get(20, 0),
            "Coverage@40":           res_full.get("CoverageByK", {}).get(40, 0),
            "TailCoverage@40":       res_full.get("TailCoverageByK", {}).get(40, 0),
            "AvgPopularity@40":      res_full.get("AvgPopularityByK", {}).get(40, 0),
            "Recall@K":              res_full["OVERALL"]["Recall@20"],
            "NDCG@K":                res_full["OVERALL"]["NDCG@20"],
        })

        if score > best_score:
            best_score   = score
            is_best      = True
            patience_cnt = 0
            print(f"  NEW BEST: NDCG@20={score:.4f}")
        else:
            patience_cnt += 1
            print(f"  Patience {patience_cnt}/{CFG['PATIENCE']}")

    history.append({"epoch": epoch+1, "loss": avg_loss, **metrics})
    ckpt_mgr.save(model, optimizer, epoch+1, metrics, history, CFG, is_best=is_best)

    # Early stopping is applied only after an evaluated joint-training epoch.
    if (not is_warmup) and patience_cnt >= CFG["PATIENCE"]:
        print(f"\n[STOP] Early stopping at epoch {epoch+1}.")
        break

print(f"\nTRAINING COMPLETE! Best NDCG@20={best_score:.4f}")



[INIT] Training BLaIR Intra-layer Gate LightGCN (Epoch 15/49)
[INFO] Each epoch uses torch.randperm — every train edge visited once per epoch.
[INFO] Two-phase training:
       Phase 1 warm-up: epoch 1 → 3, loss = L_align + L2
       Phase 2 joint:   epoch 4 → 49, loss = BPR + lambda_align*L_align + L2
[WARN] Exact intra-layer fusion is heavy: adapter+gate for all nodes is recomputed each batch.
[INFO] Negative sampling: popularity | eta=0.5 | uniform_mix=0.3 | warm-only=True


Ep 15 [JOINT-BPR]:   0%|                                               | 0/13964281 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [15]:
gc.collect(); torch.cuda.empty_cache()

In [19]:
print("\n--- LOAD BEST MODEL ---")
if os.path.exists(ckpt_mgr.best_path):
    ckpt = torch.load(ckpt_mgr.best_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    print(f"[OK] Loaded best model")
else:
    print("[WARN] No best model found, using current weights.")

print("\n--- LOADING TEST SET ---")
df_test = load_dataset(
    "chuongdo1104/amazon-2023-bronze",
    data_dir="bronze/bronze_test.parquet",
    split="train"
).to_pandas()
u_col = next(c for c in ["user_idx","user_id","reviewer_id"] if c in df_test.columns)
i_col = next(c for c in ["item_idx","item_id","parent_asin"] if c in df_test.columns)
if df_test[u_col].dtype == object:
    print("[INFO] Mapping string IDs to integer IDs...")
    df_user_map = pd.read_parquet(download_hf("gold/gold_user_id_map.parquet"))
    df_item_map = pd.read_parquet(download_hf("gold/gold_item_id_map.parquet"))
    u_map = dict(zip(df_user_map["reviewer_id"], df_user_map["user_idx"]))
    i_map = dict(zip(df_item_map["parent_asin"], df_item_map["item_idx"]))
    df_test["user_idx"] = df_test[u_col].map(u_map)
    df_test["item_idx"] = df_test[i_col].map(i_map)
    df_test = df_test.dropna(subset=["user_idx","item_idx"])

test_users_t = torch.tensor(df_test["user_idx"].values.astype(int), dtype=torch.long)
test_items_t = torch.tensor(df_test["item_idx"].values.astype(int), dtype=torch.long)

# Test positives: giữ tất cả, kể cả cold items.
# Chỉ lọc user/item ngoài range hợp lệ.
base_mask = (
    (test_users_t >= 0) & (test_users_t < num_users) &
    (test_items_t >= 0) & (test_items_t < num_items)
)
TEST_SIZE_RAW  = int(len(test_users_t))
test_users_filt = test_users_t[base_mask].to(device)
test_items_filt = test_items_t[base_mask].to(device)
cold_test_cnt   = int((item_train_freq_cpu[test_items_filt.cpu()] == 0).sum())
print(f"[FILTER] Test: {TEST_SIZE_RAW:,} -> {len(test_users_filt):,} (gồm {cold_test_cnt:,} cold positives)")

print("\n--- TEST MASKING ---")

# train_user_dict: dùng khi đánh giá validation
# test_mask_dict: dùng khi đánh giá test, gồm train + validation items
# Lưu ý: tuyệt đối không đưa test item vào mask_dict, vì test item là đáp án cần tìm.

train_user_dict = {
    int(u): items.detach().cpu().long()
    for u, items in train_mask_dict.items()
}

df_val_mask = pd.DataFrame({
    "u": val_edges_t[0].detach().cpu().numpy(),
    "i": val_edges_t[1].detach().cpu().numpy(),
})

val_user_dict = df_val_mask.groupby("u")["i"].apply(
    lambda x: torch.tensor(x.values, dtype=torch.long, device="cpu")
).to_dict()
val_user_dict = {
    int(u): items.detach().cpu().long()
    for u, items in val_user_dict.items()
}
del df_val_mask

test_mask_dict = {
    u: items.clone()
    for u, items in train_user_dict.items()
}

for u, val_items in val_user_dict.items():
    if u in test_mask_dict:
        test_mask_dict[u] = torch.unique(torch.cat([test_mask_dict[u], val_items]))
    else:
        test_mask_dict[u] = val_items.clone()

print(f"[OK] Train mask dict      : {len(train_user_dict):,} users.")
print(f"[OK] Validation mask dict : {len(val_user_dict):,} users.")
print(f"[OK] Test mask dict       : {len(test_mask_dict):,} users. Test mask = train + validation.")

item_groups_eval = item_pop_group_t.to(device)

val_cold_count  = int((item_train_freq_cpu[val_edges_t[1].cpu()] == 0).sum().item())
test_cold_count = int((item_train_freq_cpu[test_items_filt.cpu()] == 0).sum().item())
print(f"[INFO] Cold positives in validation : {val_cold_count:,}")
print(f"[INFO] Cold positives in test       : {test_cold_count:,}")
print(f"[INFO] Warm items in catalog        : {num_warm_items:,}")
print(f"[INFO] Cold items in catalog        : {num_cold_items:,} (kept in candidate set)")

gc.collect(); torch.cuda.empty_cache()


--- LOAD BEST MODEL ---
[OK] Loaded best model

--- LOADING TEST SET ---
[INFO] Mapping string IDs to integer IDs...
[INFO] Downloading gold/gold_user_id_map.parquet...
[INFO] Downloading gold/gold_item_id_map.parquet...
[FILTER] Test: 1,847,662 -> 1,847,662 (gồm 89,909 cold positives)

--- TEST MASKING ---
[OK] Train mask dict      : 1,847,662 users.
[OK] Validation mask dict : 1,847,662 users.
[OK] Test mask dict       : 1,847,662 users. Test mask = train + validation.
[INFO] Cold positives in validation : 72,685
[INFO] Cold positives in test       : 89,909
[INFO] Warm items in catalog        : 1,042,121
[INFO] Cold items in catalog        : 567,891 (kept in candidate set)


In [20]:
print("\n--- BUILDING FINAL REPRESENTATIONS ---")
model.eval()
gc.collect(); torch.cuda.empty_cache()

with torch.no_grad():
    with torch.amp.autocast("cuda", enabled=(device=="cuda")):
        h_all, zG_all, zL_all, alpha = model.forward_all(
            sparse_adj, user_blair_np, item_blair_np, node_deg_norm_t,
            device=device, chunk_size=CFG["TEXT_CHUNK_SIZE"]
        )
    user_final_dev, item_final_all = h_all.split([num_users, num_items])
    user_final_all = user_final_dev.cpu()
    del h_all, zG_all, zL_all, user_final_dev

gc.collect(); torch.cuda.empty_cache()
print(f"[OK] Representations ready. alpha={float(alpha.detach().cpu()):.4f}")

# Flat arrays — interaction-level: mỗi (user, item) là một mẫu độc lập
test_u_idx = test_users_filt.cpu().numpy().astype("int64")
test_i_idx = test_items_filt.cpu().numpy().astype("int64")
print(f"[TEST] {len(test_u_idx):,} interactions | Items: {num_items:,} (warm={num_warm_items:,} cold={num_cold_items:,})")

POP_NAMES_TEST = {2: "Item_TAIL", 3: "Item_COLD"}


def run_final_evaluation(Ks=(20, 40), user_batch=128):
    """
    Full-ranking INTERACTION-LEVEL test evaluation.

    Protocol:
    - Mỗi (user, positive_item) là một mẫu đánh giá độc lập.
    - Candidate set: toàn bộ catalog (kể cả cold items).
    - Mask: train + validation items bị loại khỏi ranking.
    - Recall@K / NDCG@K: trung bình qua toàn bộ interactions.
    - TAIL/COLD: lọc interactions có positive item thuộc nhóm đó.
    """
    Ks   = sorted(list(Ks))
    maxK = max(Ks)
    n_test = len(test_u_idx)

    item_groups_dev = item_pop_group_t.to(device)
    group_names     = ["OVERALL", "Item_TAIL", "Item_COLD"]
    accum = {
        g: {K: {"hit": 0.0, "ndcg": 0.0, "n": 0} for K in Ks}
        for g in group_names
    }
    recommended_mask = {
        K: torch.zeros(num_items, dtype=torch.bool, device=device) for K in Ks
    }
    tail_mask_eval = (item_groups_dev == 2)
    mask_value     = torch.finfo(torch.float32).min

    print(f"\n--- FULL-RANKING TEST INTERACTION-LEVEL (Ks={Ks}) ---")
    print(f"Test: {n_test:,} interactions | Items: {num_items:,} (warm={num_warm_items:,} cold={num_cold_items:,})")

    for start in tqdm(range(0, n_test, user_batch), desc="Test Eval", ncols=100):
        end    = min(start + user_batch, n_test)
        bu_np  = test_u_idx[start:end]
        bi_np  = test_i_idx[start:end]
        bu_cpu = torch.tensor(bu_np, dtype=torch.long)
        bi     = torch.tensor(bi_np, dtype=torch.long, device=device)

        u_embs = user_final_all[bu_cpu].to(device)

        with torch.no_grad():
            with torch.amp.autocast("cuda", enabled=(device=="cuda")):
                scores = torch.mm(u_embs, item_final_all.t())

        # Mask train + validation items (không mask test positives)
        for j, uid in enumerate(bu_np):
            seen = test_mask_dict.get(int(uid))
            if seen is not None:
                scores[j, seen.to(device)] = mask_value

        # Rank của positive item (1-indexed)
        pos_sc = scores[torch.arange(len(bu_cpu), device=device), bi]
        ranks  = (scores > pos_sc.unsqueeze(1)).sum(dim=1) + 1

        # Coverage mask
        _, topk_idx = torch.topk(scores, maxK, dim=1)
        for K in Ks:
            recommended_mask[K][topk_idx[:, :K].reshape(-1)] = True

        # Tích lũy metrics
        bi_g = item_groups_dev[bi]
        for K in Ks:
            hm = (ranks <= K).float()
            nm = torch.where(
                ranks <= K,
                1.0 / torch.log2(ranks.float() + 1.0),
                torch.zeros_like(ranks.float())
            )
            accum["OVERALL"][K]["hit"]  += hm.sum().item()
            accum["OVERALL"][K]["ndcg"] += nm.sum().item()
            accum["OVERALL"][K]["n"]    += len(bu_np)

            for code, gn in POP_NAMES_TEST.items():
                m = (bi_g == code)
                if m.any():
                    accum[gn][K]["hit"]  += hm[m].sum().item()
                    accum[gn][K]["ndcg"] += nm[m].sum().item()
                    accum[gn][K]["n"]    += m.sum().item()

        del u_embs, scores

    # Aggregate
    results = {}
    for g in group_names:
        results[g] = {}
        for K in Ks:
            n = accum[g][K]["n"]
            results[g][f"Recall@{K}"] = accum[g][K]["hit"]  / max(n, 1)
            results[g][f"NDCG@{K}"]   = accum[g][K]["ndcg"] / max(n, 1)
            results[g][f"n@{K}"]      = n

    # Coverage
    n_tail = int(tail_mask_eval.sum().item())
    cov, tcov, apop = {}, {}, {}
    for K in Ks:
        rm  = recommended_mask[K]
        nr  = int(rm.sum().item())
        cov[K]  = nr / num_items if num_items > 0 else 0
        tcov[K] = (rm & tail_mask_eval).sum().item() / n_tail if n_tail > 0 else 0
        apop[K] = item_train_freq_t[rm].float().mean().item() if nr > 0 else 0

    results["CoverageByK"]      = cov
    results["TailCoverageByK"]  = tcov
    results["AvgPopularityByK"] = apop
    results["EvalLevel"]        = "interaction-level"

    # Print
    print("\n" + "="*90)
    print("FINAL TEST — FULL-RANKING INTERACTION-LEVEL")
    print("="*90)
    for g in ["OVERALL", "Item_TAIL", "Item_COLD"]:
        if g not in results: continue
        n = results[g].get(f"n@{Ks[0]}", 0)
        parts = [f"{g:<16} | n={n:>10,}"]
        for K in Ks:
            r  = results[g].get(f"Recall@{K}", 0)
            nd = results[g].get(f"NDCG@{K}",   0)
            parts.append(f"R@{K}={r:.4f} N@{K}={nd:.4f}")
        print(" | ".join(parts))
    for K in Ks:
        print(f"Coverage@{K}={cov[K]:.4f} | TailCov@{K}={tcov[K]:.4f} | AvgPop@{K}={apop[K]:.1f}")
    tail_r20 = results.get("Item_TAIL", {}).get("Recall@20", 0)
    tail_n20 = results.get("Item_TAIL", {}).get("NDCG@20",   0)
    cold_r20 = results.get("Item_COLD", {}).get("Recall@20", 0)
    cold_n20 = results.get("Item_COLD", {}).get("NDCG@20",   0)
    print(f"\n  [TAIL] R@20={tail_r20:.4f} N@20={tail_n20:.4f} TailCov@20={tcov.get(20,0):.4f}")
    print(f"  [COLD] R@20={cold_r20:.4f} N@20={cold_n20:.4f} (expected ~0 for Pure CF baseline)")
    print("="*90)
    return results


results_full = run_final_evaluation(Ks=(20, 40))


--- BUILDING FINAL REPRESENTATIONS ---
[OK] Representations ready. alpha=0.6552
[TEST] 1,847,662 interactions | Items: 1,610,012 (warm=1,042,121 cold=567,891)

--- FULL-RANKING TEST INTERACTION-LEVEL (Ks=[20, 40]) ---
Test: 1,847,662 interactions | Items: 1,610,012 (warm=1,042,121 cold=567,891)


Test Eval:   0%|                                                          | 0/14435 [00:00<?, ?it/s]


FINAL TEST — FULL-RANKING INTERACTION-LEVEL
OVERALL          | n= 1,847,662 | R@20=0.0181 N@20=0.0075 | R@40=0.0274 N@40=0.0093
Item_TAIL        | n=   173,508 | R@20=0.0003 N@20=0.0001 | R@40=0.0005 N@40=0.0001
Item_COLD        | n=    89,909 | R@20=0.0000 N@20=0.0000 | R@40=0.0001 N@40=0.0000
Coverage@20=0.1264 | TailCov@20=0.1025 | AvgPop@20=49.4
Coverage@40=0.2197 | TailCov@40=0.2075 | AvgPop@40=32.9

  [TAIL] R@20=0.0003 N@20=0.0001 TailCov@20=0.1025
  [COLD] R@20=0.0000 N@20=0.0000 (expected ~0 for Pure CF baseline)


In [ ]:
import json, time
import numpy as np

class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer): return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        return super().default(obj)

print("="*70)
print(f"{'FINAL REPORT — ' + RUN_VARIANT_NAME:^70}")
print("="*70)

final_metrics = {
    "variant":                RUN_VARIANT_NAME,
    "timestamp":              time.strftime("%Y-%m-%d %H:%M:%S"),
    "protocol":               "full_ranking_interaction_level",
    "eval_level":             "interaction-level",
    "ks":                     [20, 40],
    "num_items_total":        int(num_items),
    "num_warm_items":         int(num_warm_items),
    "num_cold_items":         int(num_cold_items),
    "val_size_interactions":  int(VAL_SIZE),
    "test_size_interactions": int(len(test_users_filt)),
}

if "results_full" in globals() and isinstance(results_full, dict):
    for group_name in ["OVERALL", "Item_TAIL", "Item_COLD"]:
        if group_name in results_full:
            for K in [20, 40]:
                final_metrics[f"{group_name}_Recall@{K}"] = results_full[group_name].get(f"Recall@{K}", 0)
                final_metrics[f"{group_name}_NDCG@{K}"]   = results_full[group_name].get(f"NDCG@{K}", 0)
                final_metrics[f"{group_name}_n@{K}"]      = results_full[group_name].get(f"n@{K}", 0)

    for K in [20, 40]:
        final_metrics[f"Coverage@{K}"]      = results_full.get("CoverageByK", {}).get(K, 0)
        final_metrics[f"TailCoverage@{K}"]  = results_full.get("TailCoverageByK", {}).get(K, 0)
        final_metrics[f"AvgPopularity@{K}"] = results_full.get("AvgPopularityByK", {}).get(K, 0)

with open(os.path.join(CFG["SAVE_DIR"], f"report_{RUN_VARIANT_NAME}.json"), "w") as f:
    json.dump(final_metrics, f, indent=2, cls=NpEncoder)
print(f"[OK] Report saved to {CFG['SAVE_DIR']}/report_{RUN_VARIANT_NAME}.json")

            FINAL REPORT — blair_intralayer_gate_lightgcn             
[OK] Report saved to /content/drive/MyDrive/tarecmind/weights_blair_intralayer_gate_lightgcn/report_blair_intralayer_gate_lightgcn.json


In [ ]:
import time
print("Pipeline complete. Disconnecting in 10s...")
time.sleep(10)
try:
    from google.colab import runtime
    runtime.unassign()
except:
    print("[INFO] Not in Colab.")

Pipeline complete. Disconnecting in 10s...
